# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omesh-kumar/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [32]:
rel = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query).df()

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [33]:
!pip --version

pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)


In [34]:
# Install required package
!pip -q install duckdb

In [35]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("✅ HF_TOKEN loaded successfully!")

✅ HF_TOKEN loaded successfully!


In [36]:
import duckdb

con = duckdb.connect()

# Hugging Face secret create
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("✅ DuckDB connected to Hugging Face!")

✅ DuckDB connected to Hugging Face!


In [37]:
rel = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows
0,78835655


In [38]:
print(HF_TOKEN[:10])

hf_URPSIvp


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

- One row represents the daily performance of one content item for one client on one report date.
- Table used: `fact_content_daily_performance`
- Time window used for this assignment: March 2026 (`month=2026-03`).
- My lane: Refresh / Content Opportunity Scoring.
- Goal: Analyze page performance and build features that can later be used to rank pages for refresh opportunities.

In [39]:
query = f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5
"""

con.sql(query).df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [40]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows
0,9841378,3611061


In [41]:
query = f"""
SELECT
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day,
    COUNT(DISTINCT report_date) AS total_days
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_day,last_day,total_days
0,2026-03-01,2026-03-31,31


In [42]:
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 10
"""

features_df = con.sql(query).df()
features_df

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,67
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,616
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,28
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,25
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,1756
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,1496
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,180
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,434
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,9


## Five Features

| Feature | Available at decision time? | Reason |
|---------|------------------------------|--------|
| gsc_impressions | Yes | Already available before making a refresh decision. |
| gsc_clicks | Yes | Historical performance known before the decision. |
| gsc_sum_position | Yes | Search ranking position is already observed. |
| report_date | Yes | The reporting date is known when the decision is made. |
| client_hash_id | Yes | Identifies the client and is already available in the dataset. |

## Leakage Experiment

I deliberately added a label-derived feature to demonstrate data leakage. The model performance became unrealistically high because it had access to information that would not be available at the decision time. After removing the leaked feature, the performance returned to a more realistic level. The final analysis excludes all label-derived features.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields

### Features
- clicks
- impressions
- ctr
- average_position
- page_age_days (or publish date if available)

These features are available at the decision time and can help identify pages that may need refreshing.

### Label / Proxy
- Refresh Opportunity Score (proxy based on declining clicks/impressions and visibility).

### Context
- report_date
- client_hash_id
- content_hash_id

These fields identify the content and reporting date but are not used directly as predictive features.

### Excluded
- Future performance metrics
- Any manually created target or future information

Reason:
These fields are excluded to avoid data leakage and keep the analysis realistic.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

- This dataset cannot prove causation; it only shows observed relationships.
- Client histories are unbalanced because different clients started providing data at different times.
- Some clients only have GSC data or only GA4 data.
- The analysis is limited to the selected month (March 2026).
- Results should be used for decision support rather than as proof of future outcomes.

In [43]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Small sample for demonstration
df = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
LIMIT 5000
""").df()

# Honest model
X = df[["gsc_impressions", "gsc_sum_position"]]
y = df["gsc_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest R²:", r2_score(y_test, pred))

Honest R²: 0.34872656683233383


In [44]:
# Create a leaked feature (this should NOT be used in a real model)
df["leaked_feature"] = df["gsc_clicks"]

X_leak = df[["gsc_impressions", "gsc_sum_position", "leaked_feature"]]
y = df["gsc_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Leaked R²:", r2_score(y_test, pred))

Leaked R²: 1.0


In [45]:
df.drop(columns=["leaked_feature"], inplace=True)

print("✅ Leaked feature removed.")

✅ Leaked feature removed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it

*   List item

*   List item

*   List item

*   List item

*   List item

*   List item

*   List item

*   List item
*   List item


*   List item

*   List item

*   `List item`
*   List item


*   List item




*   List item


*   List item


*   List item


*   List item


*   List item


*   List item


- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.